# MSW-Transformer: Multi-Scale Shifted Windows Transformer Networks for 12-Lead ECG Classification
Paper: https://arxiv.org/html/2306.12098

## 1. Problem and Approach
- Task: Multi-label classification of 12-lead ECG (PTB-XL, 5 tasks: 5 superclasses, 23 subclasses, 12 rhythm, 19 form, 44 all labels).
- Pain point: CNN/RNN struggle to capture subtle, multi-scale ECG patterns; vanilla Transformers are heavy and overfit small datasets.
- Proposal: Single-layer Multi-Scale Shifted Windows Transformer (MSW-TN) with three local windows at different scales; shifted windows restrict attention to non-overlapping local regions, reduce FLOPs, and capture multi-scale cues.
- Fusion: Trainable MSW-Feature fusion assigns weights to each window’s features (softmax) before classification for better aggregation.
- Interpretability: Visualize attention over ECG leads/segments; aligns with clinical cues (e.g., ST elevation in II/III/aVF for IMI).

## 2. Architecture Highlights
- Stage 1: Patch Splitting Module (PSM) on 12-lead signals (patch size 1×5) + Linear Embedding (to C=512).
- Stage 2: Single MSW-Transformer block with three shifted window scales (e.g., 1×5, 1×10, 1×20); LayerNorm + residuals; GELU MLP per window.
- Stage 3: MSW-Feature Fusion: concatenate window features, softmax weights (trainable), weighted sum, sigmoid classifier.
- Complexity: MSW self-attention complexity Ω(MSW-SA)=4·L·C^2 + 2·L·C·ΣMi (linear in L for fixed Mi) vs global MSA quadratic in L.
- Inference: Reported much smaller params/FLOPs and faster FPS than Swin-TN, Vi-TN, GTN on PTB-XL.

## 3. Results (PTB-XL, 100 Hz, 10 s ECG)
- Metrics: Macro-F1 / Samples-F1.
- 5 superclasses: 77.85 / 81.26.
- 23 subclasses: 47.57 / 68.27.
- 12 rhythm: 66.13 / 91.32.
- 19 form: 34.60 / 50.07.
- 44 all labels: 34.29 / 63.19.
- Outperforms CNN/LSTM baselines and other Transformer variants cited; fewer params and faster inference.

## 4. Implementation Notes
- Framework: PyTorch; single MSW block; dropout 0.2 on shifted window attention.
- Training: Adam, lr=1e-4 with /10 every 10 epochs, up to 50 epochs, batch 16–32.
- Input: PTB-XL 12-lead, 100 Hz, length 1000; patch size 1×5 → tokens of dim 60 → linear proj to 512.
- Window scales: three 1×Mi windows (examples given: [1,5], [1,10], [1,20]) with shift to avoid alignment artifacts; conditions Mi | (L/P).

## 5. What to Reuse Here
- If voxel tensors ≫ length, the linear-complexity local-window attention idea could inform 3D binning/attention for spectra; trainable fusion across scales resembles multi-resolution pooling.
- For interpretability, replicate attention heatmaps over mass/frag/rt bins to align with expert cues (analogous to ECG leads).
- Keep single-block depth to avoid overfitting when data is modest; lean on multi-scale windows rather than stacking layers.

In [21]:
# Treat mz_parent bins as documents and report their count
from pathlib import Path
import numpy as np

# Locate repo root by build_foundation_lcms.py

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for parent in [cur] + list(cur.parents):
        if (parent / "build_foundation_lcms.py").exists():
            return parent
    return cur

BASE_PATH = find_repo_root(Path.cwd())
VOXEL_DIR = BASE_PATH / "data" / "voxel"  # use all voxel datasets
NPZ_LIMIT = None  # use all files

npz_files = sorted(VOXEL_DIR.rglob("*.npz"))
if NPZ_LIMIT is not None:
    npz_files = npz_files[:NPZ_LIMIT]
if not npz_files:
    print("No voxel files found in", VOXEL_DIR)
else:
    all_parents = []
    for p in npz_files:
        npz = np.load(p)
        coords = npz["coords"]
        vals = npz["vals"]
        if coords.size == 0:
            print("Empty voxel file:", p.name)
            continue
        parent_bins = np.unique(coords[:, 0]).astype(int)
        all_parents.append(parent_bins)
        # document-level summary per mz_parent bin
        print(f"{p.relative_to(BASE_PATH)}: unique mz_parent bins = {parent_bins.size}")
    if all_parents:
        all_parents = np.unique(np.concatenate(all_parents))
        num_unique_mz_parent = int(all_parents.size)
        print("Total documents (unique mz_parent across loaded files):", num_unique_mz_parent)
        print("Example mz_parent bins:", all_parents[:10])
    else:
        print("No mz_parent bins found in loaded files.")

20140103_Velos1_HRV_MV_DCmin1.npz: unique mz_parent bins = 956
Total documents (unique mz_parent across loaded files): 956
Example mz_parent bins: [0 1 2 3 4 5 6 7 8 9]


In [22]:
# Summarize mz_parent coverage and suggest coarser bins (~20 docs)
import math

TARGET_DOCS = 20

def summarize_parent_bins(npz_path: Path):
    npz = np.load(npz_path)
    coords = npz["coords"]
    if coords.size == 0:
        return None
    parents = coords[:, 0].astype(int)
    return {
        "file": npz_path.name,
        "count": int(np.unique(parents).size),
        "min": int(parents.min()),
        "max": int(parents.max()),
    }

npz_files = sorted(VOXEL_DIR.rglob("*.npz"))
if NPZ_LIMIT is not None:
    npz_files = npz_files[:NPZ_LIMIT]
if not npz_files:
    print("No voxel files found in", VOXEL_DIR)
else:
    summaries = []
    for p in npz_files:
        info = summarize_parent_bins(p)
        if info:
            summaries.append(info)
            print(f"{p.relative_to(BASE_PATH)}: parents {info['count']} (min={info['min']}, max={info['max']})")
        else:
            print("Empty voxel file:", p.name)

    if summaries:
        global_min = min(s["min"] for s in summaries)
        global_max = max(s["max"] for s in summaries)
        span = global_max - global_min + 1
        coarse_bin = max(1, math.ceil(span / TARGET_DOCS))
        print("---")
        print(f"Observed parent m/z span: {global_min} to {global_max} (width {span})")
        print(f"To get ~{TARGET_DOCS} docs, try mz_parent_bin ≈ {coarse_bin} Da")
        approx_docs = math.ceil(span / coarse_bin)
        print(f"That would yield roughly {approx_docs} parent bins over the span.")
    else:
        print("No parents to summarize.")

20140103_Velos1_HRV_MV_DCmin1.npz: parents 956 (min=0, max=1444)
---
Observed parent m/z span: 0 to 1444 (width 1445)
To get ~20 docs, try mz_parent_bin ≈ 73 Da
That would yield roughly 20 parent bins over the span.


In [23]:
# Extract and list all unique mz_parent bins across selected files
npz_files = sorted(VOXEL_DIR.rglob("*.npz"))
if NPZ_LIMIT is not None:
    npz_files = npz_files[:NPZ_LIMIT]
if not npz_files:
    print("No voxel files found in", VOXEL_DIR)
else:
    parents_all = []
    for p in npz_files:
        npz = np.load(p)
        coords = npz["coords"]
        if coords.size == 0:
            print("Empty voxel file:", p.name)
            continue
        parents_all.append(coords[:, 0].astype(int))
        print(f"Loaded {p.relative_to(BASE_PATH)} -> {np.unique(coords[:, 0]).size} parent bins")
    if parents_all:
        unique_parents = np.unique(np.concatenate(parents_all)).astype(int)
        print("Total unique mz_parent bins:", unique_parents.size)
        print("mz_parent bins (sorted):")
        print(unique_parents)
    else:
        print("No parents found in loaded files.")

Loaded 20140103_Velos1_HRV_MV_DCmin1.npz -> 956 parent bins
Total unique mz_parent bins: 956
mz_parent bins (sorted):
[   0    1    2    3    4    5    6    7    8    9   10   11   12   13
   14   15   16   17   18   19   20   21   22   23   24   25   26   28
   29   30   31   32   33   34   35   36   37   38   39   40   41   42
   43   44   45   46   47   48   49   50   51   52   53   56   57   58
   59   60   61   62   63   64   65   66   67   68   69   70   71   73
   74   75   76   77   78   79   80   81   82   83   84   85   86   87
   88   89   90   91   92   93   94   95   96   97   98   99  100  101
  102  103  104  105  106  107  108  109  110  111  112  113  114  115
  116  117  118  119  120  121  122  123  124  125  126  127  128  129
  130  131  132  133  134  135  136  137  138  139  140  141  142  143
  144  145  146  147  148  149  150  151  152  153  154  155  156  157
  158  159  160  161  162  163  164  165  166  167  168  169  170  171
  172  173  174  175  176  177

In [24]:
print(np.unique(coords[:, 0]).size)
print(np.unique(coords[:, 1]).size)
print(np.unique(coords[:, 2]).size)

956
1900
5602


In [25]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


## Plan: One Transformer per mz_parent (docs) with tokenization variants
- Doc definition: each unique `mz_parent` bin is a document; build a separate model (or batch within a multitask loop) per parent bin.
- Input source: voxel `coords` / `vals` for the selected `mz_parent` across files.
- Tokenization scenarios (make one file/config per scenario):
  1) Tokens = fragment bins (rt aggregated): summarize over rt for each frag (optionally keep top-K rt or histogram buckets).
  2) Tokens = rt bins (frag aggregated): summarize over frag for each rt (optionally keep top-K frag or histogram buckets).
  3) Tokens = (frag, rt) pairs flattened: each nonzero voxel is a token; position = 2D index; value = intensity.
  4) Coarse bins: downsample frag/rt (e.g., factor 2/4) before tokenization to cap sequence length.
- Per-scenario file to generate: `docs_<scenario>.npz` with fields:
  - `parent_bin` (int), `tokens` (2D array [doc_len, d_model] or raw sparse triplets), `positions` (indices), `values` (intensity), and `meta` (e.g., frag_range, rt_range, bin_factors).
- Training loop sketch:
  - For each parent bin: load scenario file, build dataloader, train a small Transformer (or shared backbone with parent conditioning), save checkpoints per parent.
  - Keep `TARGET_SEQ_LEN` budget per scenario (truncate or sample tokens if needed).
- Next steps: choose scenario(s) to materialize first and set bin/downsample factors to keep seq length manageable.

In [ ]:
# Build scenario files: one Transformer doc per mz_parent with multiple tokenizations
import os
import re
import yaml

OUTPUT_DIR = BASE_PATH / "data" / "doc_scenarios"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_CFG = yaml.safe_load((BASE_PATH / "configs" / "label_parsing.yaml").read_text())

def find_dataset_name(path: Path) -> str | None:
    for part in path.parts[::-1]:
        if re.match(r"^(PXD|MSV)\d+", part):
            return part
    return None

def extract_label(path: Path) -> str | None:
    ds = find_dataset_name(path)
    rule = LABEL_CFG.get(ds)
    if not rule:
        return None
    delim = rule.get("delimiter", "_")
    idx = int(rule.get("index", 0))
    trunc = rule.get("truncate", None)
    parts = path.stem.split(delim)
    if 0 <= idx < len(parts):
        part = parts[idx]
        if trunc is not None:
            try:
                t = int(trunc)
                part = part[:t]
            except Exception:
                pass
        return part
    return None

SCENARIOS = [
    {"name": "frag_only", "kind": "frag"},
    {"name": "rt_only", "kind": "rt"},
    {"name": "frag_rt", "kind": "pair"},
    {"name": "frag_rt_coarse", "kind": "pair", "frag_factor": 4, "rt_factor": 8},
]

npz_files = sorted(VOXEL_DIR.rglob("*.npz"))
if NPZ_LIMIT is not None:
    npz_files = npz_files[:NPZ_LIMIT]
if not npz_files:
    print("No voxel files found in", VOXEL_DIR)
else:
    # Collect all docs: parent -> {frag, rt, val} and labels
    parents_docs = {}
    parent_labels = {}
    for p in npz_files:
        npz = np.load(p)
        coords = npz["coords"]
        vals = npz["vals"]
        if coords.size == 0:
            continue
        par = coords[:, 0].astype(int)
        frag = coords[:, 1].astype(int)
        rt = coords[:, 2].astype(int)
        label = extract_label(p)
        if label is None:
            print(f"Warning: no label for {p.relative_to(BASE_PATH)}; using 'unknown'")
            label = "unknown"
        for parent_bin in np.unique(par):
            mask = par == parent_bin
            frag_sel = frag[mask]
            rt_sel = rt[mask]
            val_sel = vals[mask]
            acc = parents_docs.setdefault(parent_bin, {"frag": [], "rt": [], "val": []})
            acc["frag"].append(frag_sel)
            acc["rt"].append(rt_sel)
            acc["val"].append(val_sel)
            # Track label per parent; warn on conflict
            prev = parent_labels.get(parent_bin)
            if prev is not None and prev != label:
                print(f"Label conflict for parent {parent_bin}: {prev} vs {label}; keeping first")
            parent_labels.setdefault(parent_bin, label)

    # Merge lists
    for parent_bin, data in parents_docs.items():
        data["frag"] = np.concatenate(data["frag"])
        data["rt"] = np.concatenate(data["rt"])
        data["val"] = np.concatenate(data["val"])

    print("Docs ready:", len(parents_docs))

    for sc in SCENARIOS:
        name = sc["name"]
        kind = sc["kind"]
        ff = sc.get("frag_factor", 1)
        rf = sc.get("rt_factor", 1)
        parent_bins = []
        tokens_idx = []
        tokens_val = []
        labels = []

        for parent_bin, data in parents_docs.items():
            frag_arr = data["frag"]
            rt_arr = data["rt"]
            val_arr = data["val"]
            label = parent_labels.get(parent_bin, "unknown")

            if kind == "frag":
                frag_binned = frag_arr // ff
                frag_max = int(frag_binned.max())
                agg = np.bincount(frag_binned, weights=val_arr, minlength=frag_max + 1)
                nz = np.nonzero(agg)[0]
                parent_bins.append(parent_bin)
                tokens_idx.append(nz.astype(np.int64))
                tokens_val.append(agg[nz].astype(val_arr.dtype))
                labels.append(label)

            elif kind == "rt":
                rt_binned = rt_arr // rf
                rt_max = int(rt_binned.max())
                agg = np.bincount(rt_binned, weights=val_arr, minlength=rt_max + 1)
                nz = np.nonzero(agg)[0]
                parent_bins.append(parent_bin)
                tokens_idx.append(nz.astype(np.int64))
                tokens_val.append(agg[nz].astype(val_arr.dtype))
                labels.append(label)

            elif kind == "pair":
                frag_binned = frag_arr // ff
                rt_binned = rt_arr // rf
                pairs = np.stack([frag_binned, rt_binned], axis=1)
                uniq_pairs, inv = np.unique(pairs, axis=0, return_inverse=True)
                agg = np.zeros(uniq_pairs.shape[0], dtype=val_arr.dtype)
                np.add.at(agg, inv, val_arr)
                parent_bins.append(parent_bin)
                tokens_idx.append(uniq_pairs.astype(np.int64))
                tokens_val.append(agg)
                labels.append(label)

        out_path = OUTPUT_DIR / f"docs_{name}.npz"
        np.savez(
            out_path,
            parent_bins=np.array(parent_bins, dtype=np.int64),
            tokens_idx=np.array(tokens_idx, dtype=object),
            tokens_val=np.array(tokens_val, dtype=object),
            labels=np.array(labels, dtype=object),
            kind=kind,
            frag_factor=ff,
            rt_factor=rf,
        )
        print(f"Wrote {name} ->", out_path)

Docs ready: 956
Wrote frag_only -> /home/simonp/FoundationMSMS/data/doc_scenarios/docs_frag_only.npz
Wrote rt_only -> /home/simonp/FoundationMSMS/data/doc_scenarios/docs_rt_only.npz
Wrote frag_rt -> /home/simonp/FoundationMSMS/data/doc_scenarios/docs_frag_rt.npz
Wrote frag_rt_coarse -> /home/simonp/FoundationMSMS/data/doc_scenarios/docs_frag_rt_coarse.npz


In [27]:
# Ensure project src is on sys.path for foundationmsms imports
import sys
sys.path.insert(0, str(BASE_PATH / "src"))


In [28]:
# Quick training loop for MSWTransformer on frag-only docs (masked MSE reconstruction)
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from ..models import MSWTransformer, MSWConfig

SCENARIO_FILE = OUTPUT_DIR / "docs_frag_only.npz"  # produced by previous cell
EPOCHS = 1
BATCH_SIZE = 8
LR = 1e-3
MAX_STEPS = 200  # cap for quick run

# Load scenario
sc = np.load(SCENARIO_FILE, allow_pickle=True)
parent_bins = sc["parent_bins"]
tokens_idx = sc["tokens_idx"]
tokens_val = sc["tokens_val"]

# Build dataset
class FragDocDataset(Dataset):
    def __init__(self, idx_list, val_list):
        self.idx_list = idx_list
        self.val_list = val_list
        # vocab size from max token id +1 (reserve 0 for pad)
        self.vocab = int(max(int(x.max()) if len(x) else 0 for x in idx_list) + 1)

    def __len__(self):
        return len(self.idx_list)

    def __getitem__(self, i):
        idx = torch.as_tensor(self.idx_list[i], dtype=torch.long)
        val = torch.as_tensor(self.val_list[i], dtype=torch.float32)
        return idx, val

def collate(batch):
    idxs, vals = zip(*batch)
    max_len = max(x.shape[0] for x in idxs)
    padded_idx = torch.zeros(len(batch), max_len, dtype=torch.long)
    padded_val = torch.zeros(len(batch), max_len, dtype=torch.float32)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, (idx, val) in enumerate(zip(idxs, vals)):
        L = idx.shape[0]
        padded_idx[i, :L] = idx
        padded_val[i, :L] = val
        mask[i, :L] = True
    return padded_idx, padded_val, mask

dataset = FragDocDataset(tokens_idx, tokens_val)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

# Model: embedding + MSW + linear head predicting intensity
cfg = MSWConfig(dim=128, num_heads=4, window_sizes=(5, 10, 20))
model = MSWTransformer(cfg)
embed = nn.Embedding(dataset.vocab + 1, cfg.dim, padding_idx=0)
head = nn.Linear(cfg.dim, 1)

params = list(model.parameters()) + list(embed.parameters()) + list(head.parameters())
opt = torch.optim.Adam(params, lr=LR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
embed.to(device)
head.to(device)

step = 0
model.train()
for epoch in range(EPOCHS):
    for batch in dataloader:
        idx, val, mask = batch
        idx = idx.to(device)
        val = val.to(device)
        mask = mask.to(device)

        tok = embed(idx)  # (B, L, D)
        out = model(tok)  # (B, L, D)
        pred = head(out).squeeze(-1)  # (B, L)

        masked_pred = pred[mask]
        masked_val = val[mask]
        loss = torch.mean((masked_pred - masked_val) ** 2)

        opt.zero_grad()
        loss.backward()
        opt.step()

        step += 1
        if step % 20 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f}")
        if step >= MAX_STEPS:
            break
    if step >= MAX_STEPS:
        break

print("Training finished. Last loss:", loss.item())


epoch 0 step 20 loss 2127.2537
epoch 0 step 40 loss 1895.9900
epoch 0 step 60 loss 913.4482
epoch 0 step 80 loss 1169.9209
epoch 0 step 100 loss 1776.6108
epoch 0 step 120 loss 1932.1588
Training finished. Last loss: 1932.1588134765625
